# Block 1 & 2: UMAP Visualization + SHAP Feature Importance

This notebook walks through the two qualitative evaluation blocks:

| Block | Method | Research Question |
|-------|--------|-------------------|
| 1 | PCA + UMAP Visualization | Do embeddings cluster by economically meaningful categories? |
| 2 | SHAP Feature Importance | Which features drive similarity judgments? |

Results are loaded from pre-computed files in `results/` where available, with code cells that can re-run the full pipeline if a model checkpoint is provided.

In [ ]:
import sys
from pathlib import Path

# Resolve project root whether Jupyter is launched from project root
# (standard: `uv run jupyter notebook`) or from the notebooks/ sub-dir.
_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

RESULTS_DIR = PROJECT_ROOT / "results"
UMAP_DIR    = RESULTS_DIR / "figures" / "umap"
# shap_test2 is the most recent complete SHAP run; update to results/shap/ once canonical
SHAP_DIR    = RESULTS_DIR / "shap_test2"

print(f"Project root : {PROJECT_ROOT}")
print(f"UMAP results : {UMAP_DIR}")
print(f"SHAP results : {SHAP_DIR}")

---
## Block 1 — UMAP Visualization

**Methodology:**
1. Features loaded for a target period (COVID pre-crisis: Jan 2019 – Jan 2020)
2. PCA (20 components) applied for noise reduction — **all clustering metrics computed here**
3. UMAP (2D, cosine metric) applied for visual layout only
4. Points coloured by GICS sector and liquidity tier

### 1.1 Clustering Metrics (PCA space)

In [ ]:
metrics_path = UMAP_DIR / "clustering_metrics.csv"
raw_metrics = pd.read_csv(metrics_path)

# De-duplicate identical rows that can appear from multiple saves
metrics_df = raw_metrics.drop_duplicates().reset_index(drop=True)

# Keep one representative row per period (last saved)
if "period" in metrics_df.columns:
    metrics_df = metrics_df.groupby("period").last().reset_index()

metrics_df

In [ ]:
# --- Silhouette score comparison: model vs. random baseline ---

silhouette_cols = {
    "sector"     : ("silhouette_sector",      "silhouette_sector_random"),
    "liquidity"  : ("silhouette_liquidity",   "silhouette_liquidity_random"),
    "market cap" : ("silhouette_market_cap",  "silhouette_market_cap_random"),
}

row = metrics_df.iloc[-1]   # most recent period
period_label = row.get("period", "covid_pre")

labels, model_vals, rand_vals = [], [], []
for label, (col_model, col_rand) in silhouette_cols.items():
    if col_model in row and col_rand in row:
        labels.append(label)
        model_vals.append(float(row[col_model]))
        rand_vals.append(float(row[col_rand]))

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - width/2, model_vals, width, label="Model",  color="#2E86AB")
bars2 = ax.bar(x + width/2, rand_vals,  width, label="Random", color="#A23B72", alpha=0.7)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Silhouette Score")
ax.set_title(f"Silhouette Scores vs. Random Baseline  ({period_label})", fontweight="bold")
ax.legend()

for bar in bars1:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Silhouette > 0  → embeddings separate classes better than random.")
print("  Silhouette ≈ 0  → no clear cluster structure.")
print(f"  Sector silhouette = {row.get('silhouette_sector', float('nan')):.4f}")

In [ ]:
# Davies-Bouldin and Calinski-Harabász (sector, lower / higher is better respectively)
if "davies_bouldin_sector" in metrics_df.columns:
    print(f"Davies-Bouldin (sector, lower=better) : {row['davies_bouldin_sector']:.4f}")
if "calinski_harabasz_sector" in metrics_df.columns:
    print(f"Calinski-Harabász (sector, higher=better): {row['calinski_harabasz_sector']:.4f}")

### 1.2 UMAP Plots — pre-computed figures

The figures below were generated by `scripts/visualization/umap_plots.py` using the COVID pre-crisis window (Jan 2019 – Jan 2020).

In [ ]:
sector_png    = UMAP_DIR / "umap_sector_covid_pre.png"
liquidity_png = UMAP_DIR / "umap_liquidity_covid_pre.png"

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, path, title in [
    (axes[0], sector_png,    "UMAP coloured by GICS Sector"),
    (axes[1], liquidity_png, "UMAP coloured by Liquidity Tier"),
]:
    if path.exists():
        img = mpimg.imread(str(path))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title, fontweight="bold", fontsize=13)
    else:
        ax.text(0.5, 0.5, f"File not found:\n{path.name}",
                ha="center", va="center", transform=ax.transAxes, color="red")
        ax.axis("off")

plt.suptitle("COVID Pre-Crisis Embeddings  (Jan 2019 – Jan 2020)", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 1.3 Re-run UMAP (optional)

Set `CHECKPOINT_PATH` and `FEATURES_PATH` to regenerate embeddings from scratch.

In [ ]:
CHECKPOINT_PATH = None   # e.g. Path("checkpoints/last.ckpt")
FEATURES_PATH   = None   # e.g. Path("data/processed/all_features.parquet")

if CHECKPOINT_PATH is not None and FEATURES_PATH is not None:
    from src.evaluation.utils.feature_loader import FeatureLoader
    from src.evaluation.visualizations.umap_visualizer import UMAPVisualizer

    loader = FeatureLoader(str(FEATURES_PATH))
    viz = UMAPVisualizer(
        feature_loader=loader,
        output_dir=UMAP_DIR,
        checkpoint_path=CHECKPOINT_PATH,
    )
    results = viz.run_full_evaluation(period_key="covid")
    print(results)
else:
    print("Skipped — set CHECKPOINT_PATH and FEATURES_PATH to re-run.")

---
## Block 2 — SHAP Feature Importance

**Methodology:**
- SHAP KernelExplainer applied to the dual-encoder's tabular similarity function
- Explains: `similarity(query, candidate) = base_value + Σ φ_i` per tabular feature
- Results aggregated across multiple queries → global importance ranking

Features analysed (15 continuous):
`market_cap`, `beta`, `idiosyncratic_vol`, `roe`, `roa`, `debt_to_equity`,
`price_to_book`, `price_to_earnings`, `operating_margin`, `profit_margin`,
`dividend_yield`, `revenue`, `net_income`, `total_assets`, `cash`

### 2.1 Global Feature Importance

In [ ]:
importance_path = SHAP_DIR / "global_importance.csv"
raw_imp = pd.read_csv(importance_path)

# Aggregate by feature (multiple query runs produce duplicate rows)
importance_df = (
    raw_imp
    .groupby("feature", as_index=False)
    .agg(mean_abs_shap=("mean_abs_shap", "mean"), pct_importance=("pct_importance", "mean"))
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

# Recompute pct after aggregation
importance_df["pct_importance"] = (
    importance_df["mean_abs_shap"] / importance_df["mean_abs_shap"].sum() * 100
)

importance_df.head(10)

In [ ]:
# --- Horizontal bar chart: mean |SHAP| per feature ---

top_n = 12
plot_df = importance_df.head(top_n)

palette = sns.color_palette("Blues_r", n_colors=top_n)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    plot_df["feature"][::-1],
    plot_df["mean_abs_shap"][::-1],
    color=palette,
)

for bar, pct in zip(bars, plot_df["pct_importance"][::-1]):
    ax.text(
        bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
        f"{pct:.1f}%", va="center", fontsize=9
    )

ax.set_xlabel("Mean |SHAP value|  (impact on cosine similarity)")
ax.set_title("Top Feature Importances (SHAP)", fontweight="bold")
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.show()

print("\nTop-3 features:")
for _, r in importance_df.head(3).iterrows():
    print(f"  {r['feature']:25s}  {r['pct_importance']:.1f}%")

In [ ]:
# --- Cumulative importance curve ---

cumulative = importance_df["pct_importance"].cumsum().values

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(cumulative) + 1), cumulative, marker="o", color="#2E86AB")
ax.axhline(80, linestyle="--", color="orange", label="80% threshold")
ax.axhline(95, linestyle="--", color="red",    label="95% threshold")

n80 = next((i+1 for i, v in enumerate(cumulative) if v >= 80), len(cumulative))
ax.axvline(n80, linestyle=":", color="orange", alpha=0.7)
ax.text(n80 + 0.2, 50, f"{n80} features\n→ 80%", fontsize=9, color="darkorange")

ax.set_xlabel("Number of features (sorted by importance)")
ax.set_ylabel("Cumulative importance (%)")
ax.set_title("Cumulative SHAP Feature Importance", fontweight="bold")
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

### 2.2 Pre-computed SHAP figures

In [ ]:
shap_summary_png  = SHAP_DIR / "figures" / "shap_summary.png"
shap_ranking_png  = SHAP_DIR / "figures" / "shap_importance_ranking.png"
shap_waterfall    = SHAP_DIR / "figures" / "shap_waterfall_CSCO.png"

def show_image(path, title, ax):
    if Path(path).exists():
        img = mpimg.imread(str(path))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title, fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"Not found: {Path(path).name}",
                ha="center", va="center", transform=ax.transAxes, color="red")
        ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
show_image(shap_summary_png, "SHAP Beeswarm Summary", axes[0])
show_image(shap_ranking_png, "SHAP Feature Ranking (bar)", axes[1])
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
show_image(shap_waterfall, "SHAP Waterfall — CSCO query", ax)
plt.tight_layout()
plt.show()

### 2.3 Re-run SHAP (optional)

Set `CHECKPOINT_PATH` and `FEATURES_PATH` to regenerate SHAP values from scratch.

In [ ]:
CHECKPOINT_PATH = None   # e.g. Path("checkpoints/last.ckpt")
FEATURES_PATH   = None   # e.g. Path("data/processed/all_features.parquet")

if CHECKPOINT_PATH is not None and FEATURES_PATH is not None:
    import torch
    from src.models.dual_encoder import DualEncoder
    from src.evaluation.feature_importance.shap_analyzer import DualEncoderExplainer
    from src.evaluation.visualizations.shap_plots import (
        plot_global_summary, plot_importance_ranking
    )

    # Load model
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    hp   = ckpt.get("hyper_parameters", {})
    model = DualEncoder(
        temporal_input_dim=hp.get("temporal_input_dim", 13),
        tabular_continuous_dim=hp.get("tabular_continuous_dim", 15),
        embedding_dim=hp.get("embedding_dim", 128),
    )
    state = ckpt.get("state_dict", ckpt)
    state = {k.replace("model.", ""): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)

    # Load features
    import polars as pl
    features_df = pl.read_parquet(str(FEATURES_PATH)).to_pandas()
    latest = features_df.sort_values("date").groupby("symbol").last().reset_index()

    n_queries = 10
    queries   = latest.sample(n_queries, random_state=42)

    explainer = DualEncoderExplainer(model=model, background_data=latest, background_size=50)
    shap_results = explainer.explain_batch(
        queries_df=queries,
        all_stocks_df=latest,
        n_candidates_per_query=30,
        output_dir=SHAP_DIR / "per_query",
    )

    print(shap_results["global_importance"].head(10))
else:
    print("Skipped — set CHECKPOINT_PATH and FEATURES_PATH to re-run.")

---
## Summary

| Block | Key Finding |
|-------|-------------|
| 1 (UMAP) | Sector silhouette ≈ 0.107 vs. random −0.032 → weak but real sector structure. Volatility profile (idiosyncratic_vol / beta) drives spatial layout more than GICS sector alone. |
| 2 (SHAP) | `beta` dominates (≈44% importance). Model learned: similar systematic risk → similar temporal dynamics → similar embedding. `market_cap` and `idiosyncratic_vol` are secondary drivers. |

**Next steps:** Block 5 (Crisis Spearman) tests whether these embedding-based relationships remain stable across regime shifts.